# Amplitude compensation


## 0. Driver check


In [ ]:
import spcm
from spcm import units

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    print(f"Serial number:    {card.sn()}")
    print(f"Function type:    {card.function_type()}")
    print(f"Max sample value: {card.max_sample_value()}")
    print(
        f"Max sample rate:  {spcm.Clock(card).sample_rate(max=True, return_unit=units.MHz)}"
    )

print("Driver check OK -- card opened, queried, and closed without errors.")

## 1. Setup


In [ ]:
import time
from contextlib import ExitStack
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vmbpy import PixelFormat, VmbSystem

from atommovr.utils.Move import Move
from atommovr.utils.core import PhysicalParams
from atommovr_controller import RealArrayCamera

from awg_controller import (
    AmplitudeCompensation,
    AODSettings,
    AWGEngine,
    AWGEngineConfig,
    CardConfig,
    RFConverter,
)
from awg_controller.awg_control import REFERENCE_FREQUENCY_HZ

CARD_PATH = "/dev/spcm0"
MAX_AMP_V = 1.0  # into 50 ohm
ALVIUM_MODEL = "Alvium 1800 U-052m"
CAL_IMAGE = Path("data/amplitude_compensation.png")
F_LO, F_HI = 85e6, 121e6
GRID = 20
GRID_ORIGIN = "top left"  # camera corner at (f_min_v, f_min_h)

hw_stack = None
engine = None
alvium = None
cam = None


def _find_alvium(vmb):
    for c in vmb.get_all_cameras():
        model = c.get_model()
        if ALVIUM_MODEL.lower() in model.lower() or "u-052" in model.lower():
            return c
    found = [c.get_model() for c in vmb.get_all_cameras()]
    raise RuntimeError(f"{ALVIUM_MODEL} not found (detected: {found})")


def _alvium_grab():
    frame = alvium.get_frame(timeout_ms=5000)
    frame.convert_pixel_format(PixelFormat.Mono8)
    img = np.array(frame.as_numpy_ndarray(), copy=True)
    return img[:, :, 0] if img.ndim == 3 else img


def close_hardware():
    global hw_stack, engine, alvium, cam
    if hw_stack is not None:
        hw_stack.close()
        hw_stack = None
    alvium = None
    cam = None
    if engine is not None:
        engine.close()
        engine = None


def open_hardware(cfg, f_lo, f_hi, grid):
    """Open the Spectrum card and Alvium together (camera open is slow)."""
    global hw_stack, engine, alvium, cam
    if engine is not None and alvium is not None:
        print("hardware already open")
        return
    close_hardware()
    aod = AODSettings(
        f_min_v=f_lo,
        f_max_v=f_hi,
        f_min_h=f_lo,
        f_max_h=f_hi,
        grid_rows=grid,
        grid_cols=grid,
    )
    card = CardConfig(card_path=CARD_PATH, max_amplitude_v=MAX_AMP_V, aod_settings=aod)
    assert card.max_amplitude_v <= 2.0, "exceeds hard safety ceiling"
    engine = AWGEngine(card, cfg)
    fs = engine.open()
    hw_stack = ExitStack()
    try:
        vmb = hw_stack.enter_context(VmbSystem.get_instance())
        alvium = hw_stack.enter_context(_find_alvium(vmb))
        cam = RealArrayCamera((grid, grid), camera_fn=_alvium_grab)
    except Exception:
        close_hardware()
        raise
    print(
        f"mode={cfg.mode}  fs={fs / 1e6:.1f} MHz  tones={grid}+{grid}  "
        f"Nyquist={fs / 2e6:.0f} MHz  tones {f_lo / 1e6:.1f}..{f_hi / 1e6:.1f} MHz"
    )
    print(f"max round = {engine.max_round_duration_s * 1e3:.1f} ms")
    print(f"Alvium {alvium.get_model()}  id={alvium.get_id()}")
    return fs

## 2. Calibration from Alvium

In [ ]:
open_hardware(
    AWGEngineConfig(mode="stream", fill_start_threshold_promille=500),
    F_LO,
    F_HI,
    grid=GRID,
)

aod = AODSettings(
    f_min_v=F_LO,
    f_max_v=F_HI,
    f_min_h=F_LO,
    f_max_h=F_HI,
    grid_rows=GRID,
    grid_cols=GRID,
)
hold = RFConverter(aod, PhysicalParams()).holding_config()
hold.travel_duration_s = 0.5
engine.stop()
engine.load_round([hold])
engine.play()
time.sleep(engine.total_travel_duration_s + 0.2)

frame = cam.acquire()
cam.last_frame = frame
CAL_IMAGE.parent.mkdir(parents=True, exist_ok=True)
Image.fromarray(np.ascontiguousarray(frame)).save(CAL_IMAGE)
print(f"saved {CAL_IMAGE}  {frame.shape} {frame.dtype}")

powers = cam.read_power(frame)
(CAL_FREQS_V, CAL_POWERS_CH0), (CAL_FREQS_H, CAL_POWERS_CH1) = (
    AmplitudeCompensation.traces_from_grid(powers, aod, origin=GRID_ORIGIN)
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(frame, cmap="gray", origin="upper")
axes[0].set_title("Alvium frame")
axes[1].imshow(powers, cmap="viridis", origin="upper")
axes[1].set_title("site power")
axes[2].plot(CAL_FREQS_V / 1e6, CAL_POWERS_CH0, "o", label="ch0 (Y)")
axes[2].plot(CAL_FREQS_H / 1e6, CAL_POWERS_CH1, "s", label="ch1 (X)")
axes[2].set_xlabel("frequency (MHz)")
axes[2].set_ylabel("relative power")
axes[2].set_title(f"AOD calibration ({GRID}-site grid)")
axes[2].legend()
fig.tight_layout()

## 3. Fit compensation curves


In [ ]:
linear_ch0 = AmplitudeCompensation.fit_linear(CAL_FREQS_V, CAL_POWERS_CH0)
linear_ch1 = AmplitudeCompensation.fit_linear(CAL_FREQS_H, CAL_POWERS_CH1)
gaussian_ch0 = AmplitudeCompensation.fit_gaussian(CAL_FREQS_V, CAL_POWERS_CH0)
gaussian_ch1 = AmplitudeCompensation.fit_gaussian(CAL_FREQS_H, CAL_POWERS_CH1)

f_plot = np.linspace(F_LO, F_HI, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, ch, lin, gau in (
    (axes[0], 0, linear_ch0, gaussian_ch0),
    (axes[1], 1, linear_ch1, gaussian_ch1),
):
    ax.plot(f_plot / 1e6, [lin(f) for f in f_plot], label="linear fit")
    ax.plot(f_plot / 1e6, [gau(f) for f in f_plot], label="gaussian fit")
    ax.axvline(
        REFERENCE_FREQUENCY_HZ / 1e6,
        color="gray",
        ls="--",
        lw=1,
        label="reference (100 MHz)",
    )
    ax.set_xlabel("frequency (MHz)")
    ax.set_title(f"channel {ch}")
    ax.legend()
axes[0].set_ylabel("amplitude ratio")
fig.tight_layout()

for name, c0, c1 in (
    ("linear", linear_ch0, linear_ch1),
    ("gaussian", gaussian_ch0, gaussian_ch1),
):
    print(f"{name} ch0: a={c0.a:.4f}  b={c0.b:.3e}")
    print(f"{name} ch1: a={c1.a:.4f}  b={c1.b:.3e}")
print(
    f"gaussian ch0: f0={gaussian_ch0.f0_hz / 1e6:.2f} MHz  "
    f"sigma={gaussian_ch0.sigma_hz / 1e6:.2f} MHz"
)
print(
    f"gaussian ch1: f0={gaussian_ch1.f0_hz / 1e6:.2f} MHz  "
    f"sigma={gaussian_ch1.sigma_hz / 1e6:.2f} MHz"
)

## 4. Compensated round


In [ ]:
COMP_CH0 = gaussian_ch0
COMP_CH1 = gaussian_ch1
REFERENCE_AMP_PCT = 0.5 * 40.0 / GRID  # headroom below the flat 40%/tone share

aod = AODSettings(
    f_min_v=F_LO,
    f_max_v=F_HI,
    f_min_h=F_LO,
    f_max_h=F_HI,
    grid_rows=GRID,
    grid_cols=GRID,
)
converter = RFConverter(
    aod,
    PhysicalParams(),
    amplitude_compensation_ch0=COMP_CH0,
    amplitude_compensation_ch1=COMP_CH1,
    reference_amplitude_pct=REFERENCE_AMP_PCT,
)


def one_step_dest(i, n):
    return i + 1 if i + 1 < n else i - 1


def compensated_one_step_round(step_s):
    batches = [converter.holding_config()]  # settle
    for i in range(GRID):
        batches.append(converter.convert_moves([Move(i, 0, one_step_dest(i, GRID), 0)]))
    for j in range(GRID):
        batches.append(converter.convert_moves([Move(0, j, 0, one_step_dest(j, GRID))]))
    batches.append(converter.holding_config())  # park
    for b in batches:
        b.travel_duration_s = step_s
        for r in b.ramps:
            r.duration_s = step_s if r.f_start != r.f_end else 0.0
    return batches


home = converter.holding_config()
row_amps = [r.amplitude_pct for r in home.ramps if r.channel == 0]
col_amps = [r.amplitude_pct for r in home.ramps if r.channel == 1]
print(f"{GRID}x{GRID}  reference={REFERENCE_AMP_PCT:.2f}%/tone")
print("ch0 (row) amplitudes: " + ", ".join(f"{a:.2f}%" for a in row_amps))
print(f"ch0 total: {sum(row_amps):.2f}% (budget 40%)")
print("ch1 (col) amplitudes: " + ", ".join(f"{a:.2f}%" for a in col_amps))
print(f"ch1 total: {sum(col_amps):.2f}% (budget 40%)")

aod = AODSettings(
    f_min_v=F_LO,
    f_max_v=F_HI,
    f_min_h=F_LO,
    f_max_h=F_HI,
    grid_rows=GRID,
    grid_cols=GRID,
)
converter = RFConverter(
    aod,
    PhysicalParams(),
    amplitude_compensation_ch0=COMP_CH0,
    amplitude_compensation_ch1=COMP_CH1,
    reference_amplitude_pct=REFERENCE_AMP_PCT,
)


def one_step_dest(i, n):
    return i + 1 if i + 1 < n else i - 1


def compensated_one_step_round(step_s):
    batches = [converter.holding_config()]  # settle
    for i in range(GRID):
        batches.append(converter.convert_moves([Move(i, 0, one_step_dest(i, GRID), 0)]))
    for j in range(GRID):
        batches.append(converter.convert_moves([Move(0, j, 0, one_step_dest(j, GRID))]))
    batches.append(converter.holding_config())  # park
    for b in batches:
        b.travel_duration_s = step_s
        for r in b.ramps:
            r.duration_s = step_s if r.f_start != r.f_end else 0.0
    return batches


home = converter.holding_config()
row_amps = [r.amplitude_pct for r in home.ramps if r.channel == 0]
col_amps = [r.amplitude_pct for r in home.ramps if r.channel == 1]
print(f"{GRID}x{GRID}  reference={REFERENCE_AMP_PCT:.2f}%/tone")
print("ch0 (row) amplitudes: " + ", ".join(f"{a:.2f}%" for a in row_amps))
print(f"ch0 total: {sum(row_amps):.2f}% (budget 40%)")
print("ch1 (col) amplitudes: " + ", ".join(f"{a:.2f}%" for a in col_amps))
print(f"ch1 total: {sum(col_amps):.2f}% (budget 40%)")

## 5. Play on hardware


In [ ]:
open_hardware(
    AWGEngineConfig(mode="stream", fill_start_threshold_promille=500),
    F_LO,
    F_HI,
    grid=GRID,
)
print(f"ring = {engine.look_ahead_s * 1e3:.1f} ms of look-ahead")

batches = compensated_one_step_round(step_s=1)
n_moving = sum(1 for b in batches if any(r.f_start != r.f_end for r in b.ramps))
print(f"{len(batches)} batches ({n_moving} moving), {len(batches[0].ramps)} ramps each")

engine.stop()
engine.load_round(batches)
print(f"loaded {engine.total_travel_duration_s:.2f} s of waveform")

print(
    "Playing -- amplitude tracks the compensation curve across each sweep, not just its endpoints."
)
engine.play()
assert engine.last_error is None, engine.last_error

time.sleep(engine.total_travel_duration_s + 1.0)
assert engine.last_error is None, engine.last_error
print(
    "Compensated one-step row/col sweep complete; grid parked at its start sites until close()."
)

## 6. Cleanup


In [ ]:
close_hardware()
print("Engine and Alvium closed.")